# Session 4.2 | Bidirectional RNNs -- Learning from Both Directions

**Author:** Dr. Milan Joshi | MathCanvasMJ  
**Session:** 4 -- Recurrent Neural Networks  
**Framework:** TensorFlow / Keras

> *"I learned very early the difference between knowing the name of something and knowing something."* -- Richard Feynman

A standard RNN reads sequences left-to-right. But what if reading **both directions** gives better understanding? That's exactly what Bidirectional RNNs do.

In this notebook we will:
- Understand the **intuition** and **mathematics** behind Bidirectional RNNs
- Build and compare **unidirectional vs bidirectional** models on IMDB sentiment analysis
- Explore all four **merge modes** (concat, sum, mul, ave)
- Stack BiRNN layers for deeper architectures
- Examine dropout regularisation inside bidirectional wrappers
- Discuss when BiRNNs help -- and when they **don't**

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Why Bidirectional?](#1.-Why-Bidirectional?) | Intuition, analogies, and when it helps |
| 2 | [Merge Modes](#2.-Merge-Modes) | concat, sum, mul, ave -- how to combine directions |
| 3 | [IMDB Dataset](#3.-IMDB-Dataset) | Loading and exploring the sentiment dataset |
| 4 | [Baseline: Unidirectional RNN](#4.-Baseline----Unidirectional-RNN) | Standard SimpleRNN on IMDB |
| 5 | [Bidirectional RNN](#5.-Bidirectional-RNN) | BiRNN on IMDB -- the core experiment |
| 6 | [Merge Mode Exploration](#6.-Merge-Mode-Exploration) | Comparing all four merge strategies |
| 7 | [Stacked Bidirectional RNN](#7.-Stacked-Bidirectional-RNN) | Going deeper with multiple BiRNN layers |
| 8 | [BiRNN for Time Series](#8.-BiRNN-for-Time-Series) | Sine wave prediction -- with a caveat |
| 9 | [Dropout in Bidirectional Layers](#9.-Dropout-in-Bidirectional-Layers) | Regularisation inside the BiRNN wrapper |
| 10 | [When to Use BiRNNs](#10.-When-to-Use-Bidirectional-RNNs) | Decision guide and summary |

In [ ]:
# ==============================================================================
# IMPORTS AND GLOBAL CONFIGURATION
# All libraries needed for the entire notebook. We set random seeds for
# reproducibility across numpy, python's built-in random, and TensorFlow.
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import warnings
import os
import random

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN, Dense, Embedding, Bidirectional, Dropout
)
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ---- Reproducibility seeds ----
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

# ---- Plot style ----
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})
warnings.filterwarnings('ignore')

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print(f"Random seed        : {SEED}")
print("All imports successful. Ready to learn about Bidirectional RNNs!")

---
## 1. Why Bidirectional?

Consider the sentence: *"The bank of the ___"*

- If reading **left-to-right only**: Could be a *river bank* or a *money bank*
- If we also read **right-to-left** (seeing "...river was beautiful"): Clearly a *river bank*!

> **Feynman Analogy**: Reading a mystery novel -- you understand clues better on the second read because you know the ending. A BiRNN gives the network this "second read" for free.

A Bidirectional RNN processes the input sequence in **two independent passes**:

1. **Forward RNN** ($\overrightarrow{h}$): reads $x_1, x_2, \dots, x_T$ left to right  
2. **Backward RNN** ($\overleftarrow{h}$): reads $x_T, x_{T-1}, \dots, x_1$ right to left  

The two hidden states are then **merged** (concatenated by default) at each timestep to produce the final representation.

### When Does Bidirectional Help?
- **Text classification**: Full sentence is available at prediction time -- both directions are useful
- **Named Entity Recognition**: Context from both sides helps identify entities  
  (*"Paris" after "flew to" vs before "Hilton"*)
- **Speech recognition**: Future phonemes help disambiguate current ones

### When Does Bidirectional NOT Help?
- **Real-time forecasting**: You can't see the future during prediction
- **Online streaming**: Data arrives one step at a time
- **Autoregressive generation**: Each token is generated conditioned only on past tokens

### Mathematical Formulation

**Forward pass** (left $\rightarrow$ right):
$$\overrightarrow{h_t} = \tanh\left(W_{x\overrightarrow{h}} \, x_t + W_{\overrightarrow{h}\overrightarrow{h}} \, \overrightarrow{h}_{t-1} + b_{\overrightarrow{h}}\right)$$

**Backward pass** (right $\rightarrow$ left):
$$\overleftarrow{h_t} = \tanh\left(W_{x\overleftarrow{h}} \, x_t + W_{\overleftarrow{h}\overleftarrow{h}} \, \overleftarrow{h}_{t+1} + b_{\overleftarrow{h}}\right)$$

**Combined output** (default: concatenation):
$$h_t = \left[\overrightarrow{h_t} \; ; \; \overleftarrow{h_t}\right] \in \mathbb{R}^{2 n_h}$$

where:
- $x_t \in \mathbb{R}^{n_x}$ is the input at timestep $t$
- $n_h$ is the number of hidden units in each direction
- $W_{x\overrightarrow{h}} \in \mathbb{R}^{n_h \times n_x}$ maps input to forward hidden state
- $W_{\overrightarrow{h}\overrightarrow{h}} \in \mathbb{R}^{n_h \times n_h}$ is the forward recurrent weight matrix

### Parameter Count

For a single BiRNN layer followed by a Dense output layer:

$$\text{BiRNN params} = \underbrace{2 \times n_h(n_x + n_h + 1)}_{\text{two RNN directions}} + \underbrace{n_y(2n_h + 1)}_{\text{Dense layer (concat mode)}}$$

**Key insight:** BiRNN has **exactly twice** the recurrent parameters of a unidirectional RNN, plus extra output parameters due to the doubled hidden size (in concat mode).

In [ ]:
# ==============================================================================
# ARCHITECTURE DIAGRAM: BIDIRECTIONAL RNN
# We draw the forward path in BLUE (left-to-right), backward path in RED
# (right-to-left), and merge nodes in GREEN at each timestep.
# ==============================================================================

fig, ax = plt.subplots(1, 1, figsize=(14, 7))
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-0.5, 5.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Bidirectional RNN Architecture\n(5 Timesteps)', fontsize=16, fontweight='bold')

T = 5  # number of timesteps
x_positions = np.linspace(0.5, 4.5, T)

# Y-levels for input, forward, backward, merge, output
y_input = 0.5
y_fwd   = 2.0
y_bwd   = 3.2
y_merge = 4.5

box_w, box_h = 0.55, 0.45

for i, x in enumerate(x_positions):
    # ---- Input nodes ----
    rect_in = plt.Rectangle((x - box_w/2, y_input - box_h/2), box_w, box_h,
                            fc='#FFF3E0', ec='black', lw=1.5, zorder=3)
    ax.add_patch(rect_in)
    ax.text(x, y_input, f'$x_{{{i+1}}}$', ha='center', va='center', fontsize=13, zorder=4)

    # ---- Forward hidden nodes (BLUE) ----
    rect_fwd = plt.Rectangle((x - box_w/2, y_fwd - box_h/2), box_w, box_h,
                             fc='#BBDEFB', ec='#1565C0', lw=2, zorder=3)
    ax.add_patch(rect_fwd)
    ax.text(x, y_fwd, r'$\overrightarrow{h}_{' + str(i+1) + '}$',
            ha='center', va='center', fontsize=12, color='#0D47A1', zorder=4)

    # ---- Backward hidden nodes (RED) ----
    rect_bwd = plt.Rectangle((x - box_w/2, y_bwd - box_h/2), box_w, box_h,
                             fc='#FFCDD2', ec='#C62828', lw=2, zorder=3)
    ax.add_patch(rect_bwd)
    ax.text(x, y_bwd, r'$\overleftarrow{h}_{' + str(i+1) + '}$',
            ha='center', va='center', fontsize=12, color='#B71C1C', zorder=4)

    # ---- Merge nodes (GREEN) ----
    circle_merge = plt.Circle((x, y_merge), 0.22, fc='#C8E6C9', ec='#2E7D32', lw=2, zorder=3)
    ax.add_patch(circle_merge)
    ax.text(x, y_merge, f'$h_{{{i+1}}}$', ha='center', va='center', fontsize=12,
            color='#1B5E20', zorder=4)

    # ---- Vertical arrows: input -> forward, input -> backward ----
    ax.annotate('', xy=(x, y_fwd - box_h/2), xytext=(x, y_input + box_h/2),
                arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5))
    ax.annotate('', xy=(x, y_bwd - box_h/2), xytext=(x, y_input + box_h/2),
                arrowprops=dict(arrowstyle='->', color='#C62828', lw=1.5))

    # ---- Vertical arrows: forward -> merge, backward -> merge ----
    ax.annotate('', xy=(x - 0.08, y_merge - 0.22), xytext=(x - 0.08, y_fwd + box_h/2),
                arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5))
    ax.annotate('', xy=(x + 0.08, y_merge - 0.22), xytext=(x + 0.08, y_bwd + box_h/2),
                arrowprops=dict(arrowstyle='->', color='#C62828', lw=1.5))

    # ---- Horizontal arrows: forward (BLUE, left-to-right) ----
    if i < T - 1:
        ax.annotate('', xy=(x_positions[i+1] - box_w/2, y_fwd),
                    xytext=(x + box_w/2, y_fwd),
                    arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2.5))

    # ---- Horizontal arrows: backward (RED, right-to-left) ----
    if i > 0:
        ax.annotate('', xy=(x_positions[i-1] + box_w/2, y_bwd),
                    xytext=(x - box_w/2, y_bwd),
                    arrowprops=dict(arrowstyle='->', color='#C62828', lw=2.5))

# ---- Legend ----
legend_elements = [
    mpatches.Patch(fc='#BBDEFB', ec='#1565C0', lw=2, label='Forward (left-to-right)'),
    mpatches.Patch(fc='#FFCDD2', ec='#C62828', lw=2, label='Backward (right-to-left)'),
    mpatches.Patch(fc='#C8E6C9', ec='#2E7D32', lw=2, label='Merged output'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11, framealpha=0.9)

# ---- Layer labels on the left ----
ax.text(-0.3, y_input, 'Input', ha='center', va='center', fontsize=11, fontstyle='italic')
ax.text(-0.3, y_fwd,   'Forward\nRNN', ha='center', va='center', fontsize=11,
        fontstyle='italic', color='#1565C0')
ax.text(-0.3, y_bwd,   'Backward\nRNN', ha='center', va='center', fontsize=11,
        fontstyle='italic', color='#C62828')
ax.text(-0.3, y_merge,  'Merge\n(concat)', ha='center', va='center', fontsize=11,
        fontstyle='italic', color='#2E7D32')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# SIDE-BY-SIDE COMPARISON: UNIDIRECTIONAL vs BIDIRECTIONAL RNN
# Left panel: standard RNN (single direction, 4 timesteps)
# Right panel: BiRNN (two directions merged, 4 timesteps)
# ==============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

T = 4
x_pos = np.linspace(0.5, 3.5, T)

# ---- Helper to draw an RNN diagram ----
def draw_rnn(ax, title, bidirectional=False):
    ax.set_xlim(-0.5, 4.5)
    ax.set_ylim(-0.5, 5.0 if bidirectional else 4.0)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)

    bw, bh = 0.55, 0.42
    y_in  = 0.3
    y_h   = 1.6
    y_hb  = 2.8 if bidirectional else None
    y_out = (4.2 if bidirectional else 3.0)

    for i, xp in enumerate(x_pos):
        # Input
        r = plt.Rectangle((xp - bw/2, y_in - bh/2), bw, bh,
                          fc='#FFF3E0', ec='black', lw=1.3, zorder=3)
        ax.add_patch(r)
        ax.text(xp, y_in, f'$x_{{{i+1}}}$', ha='center', va='center', fontsize=12, zorder=4)

        # Forward hidden
        r = plt.Rectangle((xp - bw/2, y_h - bh/2), bw, bh,
                          fc='#BBDEFB', ec='#1565C0', lw=2, zorder=3)
        ax.add_patch(r)
        lbl = r'$\overrightarrow{h}_{' + str(i+1) + '}$' if bidirectional else f'$h_{{{i+1}}}$'
        ax.text(xp, y_h, lbl, ha='center', va='center', fontsize=11,
                color='#0D47A1', zorder=4)

        # Arrow input -> forward
        ax.annotate('', xy=(xp, y_h - bh/2), xytext=(xp, y_in + bh/2),
                    arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.3))

        # Forward horizontal
        if i < T - 1:
            ax.annotate('', xy=(x_pos[i+1] - bw/2, y_h),
                        xytext=(xp + bw/2, y_h),
                        arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))

        if bidirectional:
            # Backward hidden
            r = plt.Rectangle((xp - bw/2, y_hb - bh/2), bw, bh,
                              fc='#FFCDD2', ec='#C62828', lw=2, zorder=3)
            ax.add_patch(r)
            ax.text(xp, y_hb, r'$\overleftarrow{h}_{' + str(i+1) + '}$',
                    ha='center', va='center', fontsize=11, color='#B71C1C', zorder=4)

            # Arrow input -> backward
            ax.annotate('', xy=(xp, y_hb - bh/2), xytext=(xp, y_in + bh/2),
                        arrowprops=dict(arrowstyle='->', color='#C62828', lw=1.3))

            # Backward horizontal
            if i > 0:
                ax.annotate('', xy=(x_pos[i-1] + bw/2, y_hb),
                            xytext=(xp - bw/2, y_hb),
                            arrowprops=dict(arrowstyle='->', color='#C62828', lw=2))

            # Merge node
            c = plt.Circle((xp, y_out), 0.2, fc='#C8E6C9', ec='#2E7D32', lw=2, zorder=3)
            ax.add_patch(c)
            ax.text(xp, y_out, f'$h_{{{i+1}}}$', ha='center', va='center',
                    fontsize=11, color='#1B5E20', zorder=4)

            # Arrows to merge
            ax.annotate('', xy=(xp - 0.06, y_out - 0.2), xytext=(xp - 0.06, y_h + bh/2),
                        arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.2))
            ax.annotate('', xy=(xp + 0.06, y_out - 0.2), xytext=(xp + 0.06, y_hb + bh/2),
                        arrowprops=dict(arrowstyle='->', color='#C62828', lw=1.2))
        else:
            # Output node for unidirectional
            c = plt.Circle((xp, y_out), 0.2, fc='#E1BEE7', ec='#6A1B9A', lw=2, zorder=3)
            ax.add_patch(c)
            ax.text(xp, y_out, f'$y_{{{i+1}}}$', ha='center', va='center',
                    fontsize=11, color='#4A148C', zorder=4)
            ax.annotate('', xy=(xp, y_out - 0.2), xytext=(xp, y_h + bh/2),
                        arrowprops=dict(arrowstyle='->', color='#6A1B9A', lw=1.2))

draw_rnn(axes[0], 'Unidirectional RNN', bidirectional=False)
draw_rnn(axes[1], 'Bidirectional RNN', bidirectional=True)

plt.tight_layout()
plt.show()

---
## 2. Merge Modes -- How to Combine Forward and Backward

Keras' `Bidirectional` wrapper supports four merge modes that determine how $\overrightarrow{h_t}$ and $\overleftarrow{h_t}$ are combined:

| Mode | Formula | Output Dim | Use Case |
|------|---------|-----------|----------|
| **concat** (default) | $[\overrightarrow{h};\overleftarrow{h}]$ | $2n_h$ | Most information preserved |
| **sum** | $\overrightarrow{h} + \overleftarrow{h}$ | $n_h$ | Fewer params in next layer |
| **mul** | $\overrightarrow{h} \odot \overleftarrow{h}$ | $n_h$ | Captures interactions |
| **ave** | $\frac{\overrightarrow{h} + \overleftarrow{h}}{2}$ | $n_h$ | Smooth combination |

> **Feynman Insight**: Concatenation is like keeping both the forward and backward notes side-by-side. Summing is like merging them into one page -- you save space but might lose nuance. Multiplication forces the two directions to "agree" on what's important.

In [ ]:
# ==============================================================================
# MERGE MODES: 4-PANEL DIAGRAM
# For each merge mode, we draw two vectors being combined into one,
# showing the output dimension change with colour-coded boxes.
# ==============================================================================

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

modes = [
    ('concat', r'$[\vec{h}\;;\;\overleftarrow{h}]$', '2n_h', '#4CAF50'),
    ('sum',    r'$\vec{h} + \overleftarrow{h}$',      'n_h',  '#2196F3'),
    ('mul',    r'$\vec{h} \odot \overleftarrow{h}$',  'n_h',  '#FF9800'),
    ('ave',    r'$\frac{\vec{h}+\overleftarrow{h}}{2}$', 'n_h', '#9C27B0'),
]

for ax, (mode_name, formula, out_dim, color) in zip(axes, modes):
    ax.set_xlim(0, 6)
    ax.set_ylim(0, 8)
    ax.axis('off')
    ax.set_title(f'merge_mode="{mode_name}"', fontsize=12, fontweight='bold')

    # Forward vector (blue box)
    fwd_rect = plt.Rectangle((0.5, 5.5), 2, 1.2, fc='#BBDEFB', ec='#1565C0', lw=2)
    ax.add_patch(fwd_rect)
    ax.text(1.5, 6.1, r'$\overrightarrow{h}$' + '\n' + r'$(n_h)$',
            ha='center', va='center', fontsize=11, color='#0D47A1')

    # Backward vector (red box)
    bwd_rect = plt.Rectangle((3.5, 5.5), 2, 1.2, fc='#FFCDD2', ec='#C62828', lw=2)
    ax.add_patch(bwd_rect)
    ax.text(4.5, 6.1, r'$\overleftarrow{h}$' + '\n' + r'$(n_h)$',
            ha='center', va='center', fontsize=11, color='#B71C1C')

    # Operation symbol
    ax.text(3, 4.2, formula, ha='center', va='center', fontsize=14,
            fontweight='bold', color='#333333',
            bbox=dict(boxstyle='round,pad=0.4', fc='#F5F5F5', ec='gray', lw=1.5))

    # Arrows down
    ax.annotate('', xy=(2, 4.8), xytext=(1.5, 5.5),
                arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5))
    ax.annotate('', xy=(4, 4.8), xytext=(4.5, 5.5),
                arrowprops=dict(arrowstyle='->', color='#C62828', lw=1.5))

    # Output box
    out_w = 5 if mode_name == 'concat' else 2.5
    out_x = 0.5 if mode_name == 'concat' else 1.75
    out_rect = plt.Rectangle((out_x, 1.5), out_w, 1.2, fc=color, ec='black',
                             lw=2, alpha=0.3)
    ax.add_patch(out_rect)
    ax.text(out_x + out_w / 2, 2.1, f'output\n$({out_dim})$',
            ha='center', va='center', fontsize=12, fontweight='bold', color=color)

    # Arrow from operation to output
    ax.annotate('', xy=(3, 2.7), xytext=(3, 3.7),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

plt.suptitle('Bidirectional RNN Merge Modes', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. IMDB Dataset

We use the classic **IMDB movie review dataset** -- 50,000 reviews labelled as positive or negative sentiment. This is a perfect testbed for BiRNNs because:

1. **The entire review is available** at prediction time (no streaming constraint)
2. Sentiment cues can appear **anywhere** in the text -- beginning, middle, or end
3. Words near the end often **modify or reverse** earlier sentiment (e.g., "I thought the movie was going to be great... but it was terrible")

> **Feynman Note**: If you only read the first half of a review, you might think the reviewer loved the movie. The backward pass catches the twist at the end!

In [ ]:
# ==============================================================================
# LOAD AND PREPROCESS THE IMDB DATASET
# We limit vocabulary to 10,000 most frequent words, pad/truncate all
# sequences to maxlen=200, then inspect shapes and class balance.
# ==============================================================================

VOCAB_SIZE = 10_000
MAXLEN = 200

# ---- Load dataset ----
(X_train_raw, y_train), (X_test_raw, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# ---- Store raw lengths before padding (for histogram later) ----
train_lengths = [len(seq) for seq in X_train_raw]
test_lengths  = [len(seq) for seq in X_test_raw]

# ---- Pad sequences ----
X_train = pad_sequences(X_train_raw, maxlen=MAXLEN, padding='post', truncating='post')
X_test  = pad_sequences(X_test_raw,  maxlen=MAXLEN, padding='post', truncating='post')

print("=" * 60)
print("IMDB Dataset Summary")
print("=" * 60)
print(f"Vocabulary size : {VOCAB_SIZE:,}")
print(f"Max sequence len: {MAXLEN}")
print(f"X_train shape   : {X_train.shape}")
print(f"X_test  shape   : {X_test.shape}")
print(f"y_train shape   : {y_train.shape}")
print(f"y_test  shape   : {y_test.shape}")
print()

# ---- Class balance ----
pos_train = np.sum(y_train == 1)
neg_train = np.sum(y_train == 0)
print(f"Training set -- Positive: {pos_train} ({100*pos_train/len(y_train):.1f}%) | "
      f"Negative: {neg_train} ({100*neg_train/len(y_train):.1f}%)")
print(f"Test set     -- Positive: {np.sum(y_test==1)} | Negative: {np.sum(y_test==0)}")
print()

# ---- Decode and display a sample review ----
word_index = imdb.get_word_index()
reverse_word_index = {v + 3: k for k, v in word_index.items()}
reverse_word_index[0] = '<PAD>'
reverse_word_index[1] = '<START>'
reverse_word_index[2] = '<UNK>'
reverse_word_index[3] = '<UNUSED>'

sample_idx = 0
decoded_review = ' '.join([reverse_word_index.get(idx, '?') for idx in X_train_raw[sample_idx]])
print(f"Sample review (index {sample_idx}):")
print(f"Label: {'Positive' if y_train[sample_idx] == 1 else 'Negative'}")
print(f"Length: {len(X_train_raw[sample_idx])} tokens")
print(f"\nText (first 300 chars):\n{decoded_review[:300]}...")

In [ ]:
# ==============================================================================
# HISTOGRAM: SEQUENCE LENGTHS BEFORE PADDING
# This helps us understand how much information is lost by truncating
# to MAXLEN=200, and how much padding is added to short reviews.
# ==============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, lengths, label, color in zip(
    axes,
    [train_lengths, test_lengths],
    ['Training Set', 'Test Set'],
    ['#1565C0', '#C62828']
):
    ax.hist(lengths, bins=60, color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.axvline(x=MAXLEN, color='red', linestyle='--', linewidth=2,
               label=f'MAXLEN = {MAXLEN}')
    ax.set_xlabel('Sequence Length (tokens)')
    ax.set_ylabel('Number of Reviews')
    ax.set_title(f'{label} — Review Length Distribution')
    ax.legend(fontsize=11)

    pct_truncated = 100 * sum(1 for l in lengths if l > MAXLEN) / len(lengths)
    ax.text(0.95, 0.85, f'{pct_truncated:.1f}% truncated',
            transform=ax.transAxes, ha='right', fontsize=12,
            bbox=dict(boxstyle='round', fc='yellow', alpha=0.7))

plt.suptitle('IMDB Review Lengths Before Padding', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nMedian train length: {np.median(train_lengths):.0f} tokens")
print(f"Mean train length  : {np.mean(train_lengths):.0f} tokens")
print(f"Max train length   : {np.max(train_lengths)} tokens")

---
## 4. Baseline -- Unidirectional RNN

Before jumping to BiRNNs, we need a **baseline**. We'll build the simplest possible RNN for sentiment analysis:

$$\text{Embedding}(10000, 64) \rightarrow \text{SimpleRNN}(64) \rightarrow \text{Dense}(1, \sigma)$$

This unidirectional model reads each review from start to finish and produces a single sentiment prediction. Let's see how well it does.

In [ ]:
# ==============================================================================
# BASELINE MODEL: UNIDIRECTIONAL SimpleRNN
# Architecture: Embedding(10000,64) -> SimpleRNN(64) -> Dense(1, sigmoid)
# We train for 10 epochs with binary crossentropy and Adam optimiser.
# ==============================================================================

EMBEDDING_DIM = 64
RNN_UNITS     = 64
EPOCHS        = 10
BATCH_SIZE    = 128

# ---- Build the unidirectional model ----
model_uni = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAXLEN),
    SimpleRNN(RNN_UNITS),               # only forward direction
    Dense(1, activation='sigmoid')       # binary classification
], name='Unidirectional_RNN')

model_uni.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("=" * 60)
print("BASELINE: Unidirectional SimpleRNN")
print("=" * 60)
model_uni.summary()

# ---- Train ----
print("\nTraining...")
history_uni = model_uni.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    verbose=1
)

# ---- Evaluate ----
loss_uni, acc_uni = model_uni.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss    : {loss_uni:.4f}")
print(f"Test Accuracy: {acc_uni:.4f} ({acc_uni*100:.2f}%)")

---
## 5. Bidirectional RNN

Now let's wrap the same SimpleRNN in a `Bidirectional` layer. The Keras API makes this a one-line change:

```python
# Before (unidirectional):
SimpleRNN(64)

# After (bidirectional):
Bidirectional(SimpleRNN(64))
```

With the default `merge_mode='concat'`, the output dimension doubles from 64 to 128. This means the Dense layer will have more parameters too.

**Expected outcome**: The BiRNN should achieve higher accuracy because sentiment cues at the end of the review (which the backward pass sees early) help disambiguate the overall sentiment.

In [ ]:
# ==============================================================================
# BIDIRECTIONAL SimpleRNN MODEL
# Architecture: Embedding(10000,64) -> Bidirectional(SimpleRNN(64)) -> Dense(1)
# Notice the parameter count doubles for the RNN part, and the Dense
# layer input doubles from 64 to 128 due to concatenation.
# ==============================================================================

# ---- Reset seed for fair comparison ----
tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---- Build the bidirectional model ----
model_bi = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAXLEN),
    Bidirectional(SimpleRNN(RNN_UNITS)),   # <-- the key change!
    Dense(1, activation='sigmoid')
], name='Bidirectional_RNN')

model_bi.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("=" * 60)
print("BIDIRECTIONAL SimpleRNN")
print("=" * 60)
model_bi.summary()

# ---- Compare parameter counts ----
uni_params = model_uni.count_params()
bi_params  = model_bi.count_params()
print(f"\nParameter comparison:")
print(f"  Unidirectional : {uni_params:,} params")
print(f"  Bidirectional  : {bi_params:,} params")
print(f"  Ratio (Bi/Uni) : {bi_params/uni_params:.2f}x")

# ---- Train ----
print("\nTraining...")
history_bi = model_bi.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    verbose=1
)

# ---- Evaluate ----
loss_bi, acc_bi = model_bi.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss    : {loss_bi:.4f}")
print(f"Test Accuracy: {acc_bi:.4f} ({acc_bi*100:.2f}%)")
print(f"\nImprovement over unidirectional: {(acc_bi - acc_uni)*100:+.2f}%")

In [ ]:
# ==============================================================================
# TRAINING CURVES: UNIDIRECTIONAL vs BIDIRECTIONAL (SIDE-BY-SIDE)
# Left panel: Loss curves (training & validation for both models)
# Right panel: Accuracy curves (training & validation for both models)
# ==============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

epochs_range = range(1, EPOCHS + 1)

# ---- Loss curves ----
ax1.plot(epochs_range, history_uni.history['loss'], 'b-', label='Uni - Train', linewidth=2)
ax1.plot(epochs_range, history_uni.history['val_loss'], 'b--', label='Uni - Val', linewidth=2)
ax1.plot(epochs_range, history_bi.history['loss'], 'r-', label='Bi - Train', linewidth=2)
ax1.plot(epochs_range, history_bi.history['val_loss'], 'r--', label='Bi - Val', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss: Unidirectional vs Bidirectional', fontweight='bold')
ax1.legend(fontsize=10)

# ---- Accuracy curves ----
ax2.plot(epochs_range, history_uni.history['accuracy'], 'b-', label='Uni - Train', linewidth=2)
ax2.plot(epochs_range, history_uni.history['val_accuracy'], 'b--', label='Uni - Val', linewidth=2)
ax2.plot(epochs_range, history_bi.history['accuracy'], 'r-', label='Bi - Train', linewidth=2)
ax2.plot(epochs_range, history_bi.history['val_accuracy'], 'r--', label='Bi - Val', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy: Unidirectional vs Bidirectional', fontweight='bold')
ax2.legend(fontsize=10)

plt.suptitle('Training Comparison: UniRNN vs BiRNN on IMDB',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Merge Mode Exploration

Now let's systematically compare all four merge modes. We keep everything else identical -- same embedding, same RNN units, same training setup -- and only change `merge_mode`.

**Hypothesis**:
- `concat` should perform best (most information preserved)
- `sum` and `ave` should be similar (ave = sum / 2, just a scaling)
- `mul` is the wild card -- element-wise interaction might capture useful patterns

In [ ]:
# ==============================================================================
# MERGE MODE COMPARISON
# We loop over all four merge modes, build identical BiRNN architectures,
# train each for the same number of epochs, and collect test accuracies.
# ==============================================================================

merge_modes = ['concat', 'sum', 'mul', 'ave']
merge_results = {}  # stores {mode: (test_loss, test_acc, n_params, history)}

for mode in merge_modes:
    print(f"\n{'='*60}")
    print(f"Training BiRNN with merge_mode = '{mode}'")
    print(f"{'='*60}")

    tf.random.set_seed(SEED)
    np.random.seed(SEED)

    model = Sequential([
        Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAXLEN),
        Bidirectional(SimpleRNN(RNN_UNITS), merge_mode=mode),
        Dense(1, activation='sigmoid')
    ], name=f'BiRNN_{mode}')

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    n_params = model.count_params()
    print(f"  Parameters: {n_params:,}")

    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        verbose=0
    )

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    merge_results[mode] = (test_loss, test_acc, n_params, history)
    print(f"  Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

# ---- Bar chart comparison ----
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0']
accs = [merge_results[m][1] for m in merge_modes]
bars = ax.bar(merge_modes, accs, color=colors, edgecolor='black', linewidth=1.5, width=0.6)

# Add value labels on bars
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{acc:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_xlabel('Merge Mode', fontsize=13)
ax.set_ylabel('Test Accuracy', fontsize=13)
ax.set_title('BiRNN Test Accuracy by Merge Mode', fontsize=14, fontweight='bold')
ax.set_ylim(min(accs) - 0.05, max(accs) + 0.03)

# Add baseline line
ax.axhline(y=acc_uni, color='gray', linestyle='--', linewidth=1.5,
           label=f'Unidirectional baseline: {acc_uni:.4f}')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# PARAMETER COUNT TABLE: MERGE MODES
# Shows how 'concat' doubles the Dense layer input while other modes
# keep the same dimension, leading to different total parameter counts.
# ==============================================================================

# ---- Build a DataFrame for comparison ----
table_data = []
for mode in merge_modes:
    test_loss, test_acc, n_params, _ = merge_results[mode]
    output_dim = 2 * RNN_UNITS if mode == 'concat' else RNN_UNITS
    dense_params = output_dim + 1  # weights + bias for Dense(1)
    rnn_params = n_params - VOCAB_SIZE * EMBEDDING_DIM - dense_params
    table_data.append({
        'Merge Mode': mode,
        'BiRNN Output Dim': output_dim,
        'RNN Params': f"{rnn_params:,}",
        'Dense Params': f"{dense_params:,}",
        'Total Params': f"{n_params:,}",
        'Test Accuracy': f"{test_acc:.4f}",
    })

df_merge = pd.DataFrame(table_data)
print("\n" + "=" * 80)
print("MERGE MODE PARAMETER COMPARISON")
print("=" * 80)
print(df_merge.to_string(index=False))

# ---- Matplotlib table for visual presentation ----
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')
ax.set_title('Parameter Count by Merge Mode', fontsize=14, fontweight='bold', pad=20)

col_colors = ['#E3F2FD'] * len(df_merge.columns)
table = ax.table(
    cellText=df_merge.values,
    colLabels=df_merge.columns,
    cellLoc='center',
    loc='center',
    colColours=col_colors
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.6)

# Highlight concat row
for j in range(len(df_merge.columns)):
    table[1, j].set_facecolor('#C8E6C9')

plt.tight_layout()
plt.show()

---
## 7. Stacked Bidirectional RNN

Just like stacking regular RNNs, we can stack BiRNN layers for deeper models. Each layer except the last needs `return_sequences=True`.

**Key architectural detail:** For a 2-layer stacked BiRNN with `merge_mode='concat'`, the second layer receives inputs of size $2n_h$ from the first BiRNN layer. This means:

$$\text{Layer 1 input: } n_x \xrightarrow{\text{BiRNN}} 2n_h \xrightarrow{\text{BiRNN}} 2(2n_h) = 4n_h$$

Wait -- that's not quite right! Each direction in Layer 2 still has $n_h$ units, but it receives $2n_h$-dimensional inputs. So:
- Layer 2 forward RNN: $\mathbb{R}^{2n_h} \rightarrow \mathbb{R}^{n_h}$
- Layer 2 backward RNN: $\mathbb{R}^{2n_h} \rightarrow \mathbb{R}^{n_h}$
- Layer 2 output (concat): $\mathbb{R}^{2n_h}$

> **Feynman Note**: Stacking BiRNNs is like re-reading a book multiple times -- each layer builds a higher-level understanding from the bidirectional context of the layer below.

In [ ]:
# ==============================================================================
# STACKED BIDIRECTIONAL RNN: 1-LAYER vs 2-LAYER
# The first BiRNN layer must use return_sequences=True to pass the full
# sequence of hidden states to the second BiRNN layer.
# ==============================================================================

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---- 1-Layer BiRNN (already trained above, but retrain for fairness) ----
model_1layer = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAXLEN),
    Bidirectional(SimpleRNN(RNN_UNITS)),
    Dense(1, activation='sigmoid')
], name='BiRNN_1Layer')
model_1layer.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("1-Layer BiRNN:")
print(f"  Total params: {model_1layer.count_params():,}")

history_1layer = model_1layer.fit(
    X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.2, verbose=0
)
loss_1l, acc_1l = model_1layer.evaluate(X_test, y_test, verbose=0)
print(f"  Test Accuracy: {acc_1l:.4f}")

# ---- 2-Layer Stacked BiRNN ----
tf.random.set_seed(SEED)
np.random.seed(SEED)

model_2layer = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAXLEN),
    Bidirectional(SimpleRNN(RNN_UNITS, return_sequences=True)),  # return full sequence
    Bidirectional(SimpleRNN(RNN_UNITS)),                         # final hidden state
    Dense(1, activation='sigmoid')
], name='BiRNN_2Layer_Stacked')
model_2layer.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\n2-Layer Stacked BiRNN:")
model_2layer.summary()

history_2layer = model_2layer.fit(
    X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_split=0.2, verbose=0
)
loss_2l, acc_2l = model_2layer.evaluate(X_test, y_test, verbose=0)
print(f"  Test Accuracy: {acc_2l:.4f}")

# ---- Compare training curves ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

epochs_range = range(1, EPOCHS + 1)

# Loss
ax1.plot(epochs_range, history_1layer.history['loss'], 'b-', label='1-Layer Train', lw=2)
ax1.plot(epochs_range, history_1layer.history['val_loss'], 'b--', label='1-Layer Val', lw=2)
ax1.plot(epochs_range, history_2layer.history['loss'], 'g-', label='2-Layer Train', lw=2)
ax1.plot(epochs_range, history_2layer.history['val_loss'], 'g--', label='2-Layer Val', lw=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss: 1-Layer vs 2-Layer Stacked BiRNN', fontweight='bold')
ax1.legend()

# Accuracy
ax2.plot(epochs_range, history_1layer.history['accuracy'], 'b-', label='1-Layer Train', lw=2)
ax2.plot(epochs_range, history_1layer.history['val_accuracy'], 'b--', label='1-Layer Val', lw=2)
ax2.plot(epochs_range, history_2layer.history['accuracy'], 'g-', label='2-Layer Train', lw=2)
ax2.plot(epochs_range, history_2layer.history['val_accuracy'], 'g--', label='2-Layer Val', lw=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy: 1-Layer vs 2-Layer Stacked BiRNN', fontweight='bold')
ax2.legend()

plt.suptitle('Stacked Bidirectional RNN Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n1-Layer BiRNN Test Accuracy: {acc_1l:.4f}")
print(f"2-Layer BiRNN Test Accuracy: {acc_2l:.4f}")
print(f"Difference: {(acc_2l - acc_1l)*100:+.2f}%")

---
## 8. BiRNN for Time Series

Can we use a Bidirectional RNN for time series? **Technically yes, but with an important caveat.**

> **Feynman Warning**: In real-time forecasting, you cannot see the future. A BiRNN that reads future values during training will learn patterns it can never exploit at inference time on truly unseen data.

However, for certain offline tasks (e.g., filling in missing values in a recorded signal, or smoothing), BiRNNs can outperform unidirectional models.

Let's generate a synthetic sine wave and compare:
- Unidirectional RNN: predicts next value from past values only
- Bidirectional RNN: uses both past and future context (cheating for forecasting!)

In [ ]:
# ==============================================================================
# BIRNN FOR TIME SERIES: SINE WAVE PREDICTION
# We generate a noisy sine wave, create windowed sequences, and compare
# unidirectional vs bidirectional RNNs. Note the conceptual caveat:
# BiRNN "sees the future" which is unfair for true forecasting.
# ==============================================================================

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- Generate synthetic sine wave with noise ----
n_points = 2000
t = np.linspace(0, 20 * np.pi, n_points)
signal = np.sin(t) + 0.1 * np.random.randn(n_points)

# ---- Create windowed sequences ----
window_size = 30
X_ts, y_ts = [], []
for i in range(len(signal) - window_size):
    X_ts.append(signal[i:i + window_size])
    y_ts.append(signal[i + window_size])

X_ts = np.array(X_ts).reshape(-1, window_size, 1)
y_ts = np.array(y_ts)

# ---- Train/test split ----
split = int(0.8 * len(X_ts))
X_ts_train, X_ts_test = X_ts[:split], X_ts[split:]
y_ts_train, y_ts_test = y_ts[:split], y_ts[split:]

print(f"Time series shapes: X_train={X_ts_train.shape}, X_test={X_ts_test.shape}")

# ---- Model 1: Unidirectional RNN ----
tf.random.set_seed(SEED)
model_ts_uni = Sequential([
    SimpleRNN(32, input_shape=(window_size, 1)),
    Dense(1)
], name='TimeSeries_UniRNN')
model_ts_uni.compile(optimizer='adam', loss='mse')
model_ts_uni.fit(X_ts_train, y_ts_train, epochs=20, batch_size=64, verbose=0)
pred_uni_ts = model_ts_uni.predict(X_ts_test, verbose=0).flatten()
rmse_uni_ts = np.sqrt(np.mean((pred_uni_ts - y_ts_test)**2))

# ---- Model 2: Bidirectional RNN ----
tf.random.set_seed(SEED)
model_ts_bi = Sequential([
    Bidirectional(SimpleRNN(32), input_shape=(window_size, 1)),
    Dense(1)
], name='TimeSeries_BiRNN')
model_ts_bi.compile(optimizer='adam', loss='mse')
model_ts_bi.fit(X_ts_train, y_ts_train, epochs=20, batch_size=64, verbose=0)
pred_bi_ts = model_ts_bi.predict(X_ts_test, verbose=0).flatten()
rmse_bi_ts = np.sqrt(np.mean((pred_bi_ts - y_ts_test)**2))

print(f"\nRMSE (Unidirectional) : {rmse_uni_ts:.4f}")
print(f"RMSE (Bidirectional)  : {rmse_bi_ts:.4f}")

# ---- Plot predictions ----
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
plot_range = slice(0, 200)

axes[0].plot(y_ts_test[plot_range], 'k-', label='True', linewidth=1.5, alpha=0.8)
axes[0].plot(pred_uni_ts[plot_range], 'b--', label=f'UniRNN (RMSE={rmse_uni_ts:.4f})', lw=1.5)
axes[0].set_title('Unidirectional RNN — Sine Wave Prediction', fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].set_ylabel('Amplitude')

axes[1].plot(y_ts_test[plot_range], 'k-', label='True', linewidth=1.5, alpha=0.8)
axes[1].plot(pred_bi_ts[plot_range], 'r--', label=f'BiRNN (RMSE={rmse_bi_ts:.4f})', lw=1.5)
axes[1].set_title('Bidirectional RNN — Sine Wave Prediction', fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Amplitude')

plt.suptitle('Time Series Prediction: UniRNN vs BiRNN\n'
             '(Note: BiRNN sees future context within each window — not true forecasting!)',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("IMPORTANT CAVEAT")
print("="*60)
print("The BiRNN reads the window both forwards and backwards.")
print("For TRUE forecasting (predicting unseen future), the backward")
print("pass gives no real advantage — it only sees the same window.")
print("BiRNN excels in tasks where the ENTIRE sequence is available,")
print("such as gap-filling, denoising, or classification.")

---
## 9. Dropout in Bidirectional Layers

Regularisation is critical for RNNs, which are prone to overfitting. Keras' `SimpleRNN` supports two types of dropout:

1. **`dropout`**: Applied to the **input** connections at each timestep
2. **`recurrent_dropout`**: Applied to the **recurrent** connections (hidden-to-hidden)

When wrapped in `Bidirectional`, dropout is applied **independently** to each direction. This means with `dropout=0.2`, both the forward and backward RNNs drop 20% of their input connections.

> **Feynman Analogy**: Dropout is like studying with random pages torn out of your textbook. It forces you to learn robust representations rather than memorising specific sequences of words.

In [ ]:
# ==============================================================================
# DROPOUT EXPERIMENT: TESTING DIFFERENT DROPOUT RATES WITH BiRNN
# We train BiRNN models with dropout rates [0.0, 0.2, 0.4] and compare
# train vs validation accuracy to see overfitting reduction.
# ==============================================================================

dropout_rates = [0.0, 0.2, 0.4]
dropout_results = {}

for dr in dropout_rates:
    print(f"\nTraining BiRNN with dropout={dr}...")
    tf.random.set_seed(SEED)
    np.random.seed(SEED)

    model_dr = Sequential([
        Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAXLEN),
        Bidirectional(SimpleRNN(RNN_UNITS, dropout=dr, recurrent_dropout=dr)),
        Dense(1, activation='sigmoid')
    ], name=f'BiRNN_dropout_{str(dr).replace(".", "_")}')

    model_dr.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    history_dr = model_dr.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        verbose=0
    )

    test_loss_dr, test_acc_dr = model_dr.evaluate(X_test, y_test, verbose=0)
    dropout_results[dr] = {
        'test_acc': test_acc_dr,
        'history': history_dr,
    }
    print(f"  Test Accuracy: {test_acc_dr:.4f}")

# ---- Plot: Train vs Val accuracy for each dropout rate ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
colors_train = ['#1565C0', '#2E7D32', '#C62828']
colors_val   = ['#42A5F5', '#66BB6A', '#EF5350']

for ax, dr, ct, cv in zip(axes, dropout_rates, colors_train, colors_val):
    h = dropout_results[dr]['history']
    test_acc = dropout_results[dr]['test_acc']

    ax.plot(range(1, EPOCHS+1), h.history['accuracy'], color=ct,
            linewidth=2, label='Train Acc')
    ax.plot(range(1, EPOCHS+1), h.history['val_accuracy'], color=cv,
            linewidth=2, linestyle='--', label='Val Acc')
    ax.fill_between(
        range(1, EPOCHS+1),
        h.history['accuracy'],
        h.history['val_accuracy'],
        alpha=0.15, color='red', label='Overfitting gap'
    )
    ax.set_xlabel('Epoch')
    if dr == 0.0:
        ax.set_ylabel('Accuracy')
    ax.set_title(f'Dropout = {dr}\nTest Acc = {test_acc:.4f}', fontweight='bold')
    ax.legend(fontsize=9, loc='lower right')

plt.suptitle('Effect of Dropout on BiRNN: Overfitting Reduction',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# ---- Summary ----
print("\n" + "="*60)
print("DROPOUT EXPERIMENT SUMMARY")
print("="*60)
for dr in dropout_rates:
    h = dropout_results[dr]['history']
    final_train = h.history['accuracy'][-1]
    final_val   = h.history['val_accuracy'][-1]
    gap = final_train - final_val
    print(f"  dropout={dr}: Train={final_train:.4f}, Val={final_val:.4f}, "
          f"Gap={gap:.4f}, Test={dropout_results[dr]['test_acc']:.4f}")

---
## 10. When to Use Bidirectional RNNs

### Decision Guide

| Scenario | Use BiRNN? | Reason |
|----------|-----------|--------|
| Text classification | Yes | Full text available at prediction time |
| Sentiment analysis | Yes | Context from both sides helps |
| Named Entity Recognition | Yes | Left and right context disambiguates entities |
| Machine translation | Encoder only | Decoder is autoregressive (left-to-right) |
| Question answering | Yes | Full passage is available for reading |
| Time series forecasting | No | Can't see the future during prediction |
| Real-time speech | No | Must process in real-time as audio arrives |
| Offline speech recognition | Yes | Full audio recording is available |
| Text generation | No | Autoregressive: each token depends on past only |
| Gap-filling / Denoising | Yes | Both sides of the gap provide useful context |

### Rules of Thumb

1. **"Is the entire input available at inference time?"** If yes, BiRNN can help.
2. **"Am I generating output one step at a time?"** If yes, BiRNN won't help for the decoder.
3. **"Does my task benefit from future context?"** If yes, BiRNN is a strong choice.

> **Feynman's Razor for BiRNNs**: *"If you have both ends of the string, use both ends. If you're still weaving the string, use only what you've woven so far."*

---
## Summary and Key Takeaways

### What We Learned

1. **Bidirectional RNNs** process sequences in both directions, capturing context from past AND future.

2. The `Bidirectional` wrapper in Keras makes it a **one-line change** from unidirectional to bidirectional.

3. **Merge modes** control how the two directions are combined:
   - `concat` (default): Preserves most information, doubles output dimension
   - `sum` / `ave`: Keeps same dimension, fewer downstream parameters
   - `mul`: Element-wise interaction between directions

4. **Stacking** BiRNN layers creates deeper models; intermediate layers need `return_sequences=True`.

5. **Dropout** within BiRNN helps reduce overfitting -- applied independently to each direction.

6. BiRNNs are **not suitable for real-time forecasting** where future data is unavailable.

### Performance Comparison (IMDB Sentiment)

| Model | Architecture | Test Accuracy | Params |
|-------|-------------|---------------|--------|
| Unidirectional RNN | Embed -> SimpleRNN(64) -> Dense(1) | ~73-76% | Baseline |
| Bidirectional RNN (concat) | Embed -> Bi(SimpleRNN(64)) -> Dense(1) | ~75-78% | ~1.5x |
| Stacked BiRNN (2 layers) | Embed -> Bi(RNN, ret_seq) -> Bi(RNN) -> Dense(1) | ~74-77% | ~2.5x |

*(Exact numbers depend on random initialisation and hardware)*

### What's Next?

The SimpleRNN suffers from the **vanishing gradient problem** for long sequences. In the next session:

**Session 5.1: Long Short-Term Memory (LSTM)** -- We'll see how gating mechanisms solve the vanishing gradient problem, and combine LSTM with bidirectional processing for even better results.

---
*Notebook by Dr. Milan Joshi | MathCanvasMJ*  
*Session 4.2 -- Bidirectional RNNs*  
*"What I cannot create, I do not understand."* -- Richard Feynman